# 🔍 RAG (Retrieval-Augmented Generation) Pipeline

A complete, end-to-end RAG pipeline built on Wikipedia-style documents.

**Pipeline overview:**
1. Load & explore the dataset
2. Build document chunks
3. Embed chunks with SentenceTransformers
4. Index with FAISS
5. Retrieve relevant context
6. Generate answers with a local LLM


## 1. Install Dependencies

In [ ]:
%pip install -q langchain langchain-text-splitters sentence-transformers faiss-cpu transformers torch pyarrow


## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("All imports successful ✅")


## 3. Load Dataset

Load the parquet file and preview the schema.


In [ ]:
DATA_PATH = "a.parquet/a.parquet"

df = pd.read_parquet(DATA_PATH)

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {df.columns.tolist()}")
df.head(3)


## 4. Exploratory Data Analysis

In [ ]:
# Basic stats
print("=== Dataset Info ===")
df.info()

print("\n=== Sample rows ===")
display(df.sample(3, random_state=42))


In [ ]:
# Text length distribution
df["text_len"] = df["text"].str.len()

print("=== Text length stats ===")
print(df["text_len"].describe().round(0))

print("\n=== Percentiles ===")
print(df["text_len"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).round(0))


## 5. Build Document Strings

Combine title, text, and categories into one rich document string per row.


In [ ]:
def build_document(row: pd.Series) -> str:
    """Combine structured fields into a single searchable document."""
    return (
        f"Title: {row['title']}\n"
        f"Text: {row['text']}\n"
        f"Categories: {row['categories']}"
    )

df["document"] = df.apply(build_document, axis=1)

# Sanity check
print("Sample document (first 600 chars):")
print("-" * 60)
print(df["document"].iloc[0][:600])


## 6. Text Chunking

Use `RecursiveCharacterTextSplitter` — it tries to split on paragraphs → 
sentences → words before falling back to characters, keeping chunks coherent.

| Parameter      | Value | Reason |
|---------------|-------|--------|
| `chunk_size`  | 512   | Fits most embedding models' context window |
| `chunk_overlap`| 64   | Ensures context continuity across chunk boundaries |


In [ ]:
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 64
NUM_DOCS      = 1000   # ← increase for a larger index; full dataset = ~442k docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)

sample_docs = df["document"].head(NUM_DOCS).tolist()

all_chunks: list[str] = []
for doc in sample_docs:
    all_chunks.extend(splitter.split_text(doc))

print(f"Documents processed : {NUM_DOCS:,}")
print(f"Total chunks        : {len(all_chunks):,}")
print(f"Avg chunks/doc      : {len(all_chunks)/NUM_DOCS:.1f}")

# Preview
print("\n--- Chunk 0 ---")
print(all_chunks[0])


## 7. Embed Chunks with SentenceTransformers

`all-MiniLM-L6-v2` is a lightweight, fast model (384-dim) that works well 
for semantic similarity tasks.


In [ ]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL)

print("Encoding chunks (this may take a minute) ...")
chunk_embeddings = embed_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalize here → dot product == cosine sim
)

print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")


## 8. Build FAISS Vector Index

We use `IndexFlatIP` (inner product). Because embeddings are L2-normalised, 
inner product equals cosine similarity — higher score = more relevant.


In [ ]:
embeddings_f32 = np.array(chunk_embeddings, dtype="float32")

dimension = embeddings_f32.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_f32)

print(f"Embedding dimension : {dimension}")
print(f"Vectors in index    : {index.ntotal:,}")


## 9. Retrieval Function

`search()` takes a query string, embeds it, and returns the top-k most 
similar chunks along with their cosine similarity scores.


In [ ]:
def search(query: str, k: int = 5) -> tuple[list[str], np.ndarray]:
    """
    Retrieve the top-k chunks most relevant to `query`.

    Returns
    -------
    results : list[str]
        The retrieved text chunks.
    scores : np.ndarray
        Cosine similarity scores (higher = better, max 1.0).
    """
    query_emb = embed_model.encode(
        [query],
        normalize_embeddings=True,
    )
    query_emb = np.array(query_emb, dtype="float32")

    scores, indices = index.search(query_emb, k)
    results = [all_chunks[i] for i in indices[0]]
    return results, scores[0]


def pretty_search(query: str, k: int = 3) -> None:
    """Print a nicely formatted search result."""
    results, scores = search(query, k=k)
    print(f"Query : {query!r}")
    print("=" * 70)
    for rank, (chunk, score) in enumerate(zip(results, scores), start=1):
        print(f"\n[Rank {rank}]  Score: {score:.4f}")
        print("-" * 70)
        print(chunk[:400])
    print()


In [ ]:
# Quick smoke-tests
pretty_search("football clubs in Greece")
pretty_search("Greek financial crisis")


## 10. Build RAG Prompt

Combine retrieved chunks into a context block, then wrap with a clear 
instruction prompt for the LLM.


In [ ]:
def build_prompt(query: str, context_chunks: list[str]) -> str:
    """Format a RAG prompt from a query and retrieved context chunks."""
    context = "\n\n---\n\n".join(context_chunks)
    return f"""You are a helpful assistant. Answer the question using ONLY the provided context.
If the context does not contain enough information, say "I don't know."

Context:
{context}

Question: {query}

Answer:"""


# Demo
sample_query = "What caused the Greek financial crisis?"
sample_chunks, _ = search(sample_query, k=3)
sample_prompt = build_prompt(sample_query, sample_chunks)

print(sample_prompt[:1200])


## 11. Load Local LLM

`TinyLlama-1.1B-Chat` is a small, fast chat model — good for demos on CPU.  
Swap for any HuggingFace model (e.g. `Mistral-7B-Instruct`) if you have a GPU.


In [ ]:
LLM_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading LLM: {LLM_MODEL} ...")
llm = pipeline(
    "text-generation",
    model=LLM_MODEL,
    max_new_tokens=256,
    temperature=0.1,         # low temperature → more factual answers
    do_sample=True,
    pad_token_id=2,          # suppress padding warning
)
print("LLM loaded ✅")


## 12. Full RAG Pipeline

One function that ties retrieval + generation together.


In [ ]:
def rag(query: str, k: int = 3, max_new_tokens: int = 256) -> str:
    """
    End-to-end RAG:
      1. Retrieve top-k relevant chunks.
      2. Build a prompt.
      3. Generate an answer with the LLM.

    Returns the generated answer string.
    """
    # Step 1 – Retrieve
    context_chunks, scores = search(query, k=k)

    # Step 2 – Build prompt
    prompt = build_prompt(query, context_chunks)

    # Step 3 – Generate
    messages = [{"role": "user", "content": prompt}]
    output = llm(messages, max_new_tokens=max_new_tokens)

    # Extract assistant turn from chat response
    generated = output[0]["generated_text"]
    if isinstance(generated, list):
        # pipeline returns a list of message dicts in chat mode
        answer = generated[-1].get("content", "").strip()
    else:
        answer = generated.strip()

    return answer


## 13. Run the RAG Pipeline

In [ ]:
# ── Example queries ──────────────────────────────────────────────
QUERIES = [
    "What caused the Greek financial crisis?",
    "Which football clubs are based in Greece?",
    "Who is the Assistant Secretary of Defense for Homeland Defense?",
]

for q in QUERIES:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"{'='*70}")
    answer = rag(q, k=3)
    print(f"A: {answer}")


## 14. Interactive Q&A

In [ ]:
# Run this cell to enter your own question
user_query = input("Enter your question (or press Enter to skip): ").strip()

if user_query:
    print("\nSearching ...")
    answer = rag(user_query, k=5)
    print(f"\nAnswer:\n{answer}")
else:
    print("Skipped.")


## 15. Save & Reload the Index (Optional)

Persist the FAISS index and chunk list so you don't need to re-embed next time.


In [ ]:
import pickle, pathlib

SAVE_DIR = pathlib.Path("rag_artifacts")
SAVE_DIR.mkdir(exist_ok=True)

# Save FAISS index
faiss.write_index(index, str(SAVE_DIR / "faiss.index"))

# Save chunk list
with open(SAVE_DIR / "chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print(f"Saved index + {len(all_chunks):,} chunks to {SAVE_DIR}/")

# ── Reload example ──────────────────────────────────────────────
# index2 = faiss.read_index(str(SAVE_DIR / "faiss.index"))
# with open(SAVE_DIR / "chunks.pkl", "rb") as f:
#     all_chunks2 = pickle.load(f)
# print("Reloaded:", index2.ntotal, "vectors")


---

## Summary

| Step | Component | Choice |
|------|-----------|--------|
| Chunking | RecursiveCharacterTextSplitter | 512 chars, 64 overlap |
| Embeddings | all-MiniLM-L6-v2 | 384-dim, L2-normalised |
| Vector store | FAISS IndexFlatIP | Exact cosine search |
| LLM | TinyLlama-1.1B-Chat | Lightweight, CPU-friendly |

**To scale up:**
- Increase `NUM_DOCS` toward the full 442k.
- Swap `IndexFlatIP` → `IndexIVFFlat` or `IndexHNSWFlat` for ANN search.
- Replace TinyLlama with a larger instruction-tuned model (Mistral 7B, Llama 3, etc.).
